### DataFrame transformations	select, filter, drop, withColumn

In [0]:
data = [
  (1, "Alice", 34, "NY"),
  (2, "Bob", None, "CA"),
  (3, "Cathy", 29, "TX"),
]
df = spark.createDataFrame(data, schema=["id","name","age","state"])
display(df)


In [0]:
df.select('id','name').show()

In [0]:
from pyspark.sql import functions as F
df.selectExpr('id',"upper(name) as first_name","initcap(name) as last_name",'age is not null as is_age').show()

In [0]:
# Filter and where
df.filter(F.col('age')>30).show()

In [0]:
df.where(F.col("age").isNull()).show()

In [0]:
#withColumn
df.withColumn("age_plus_1",F.coalesce(F.col('age'),F.lit(0)+1)).show()

In [0]:
assert df.filter(F.col("age")>30).count()==1

### Data cleaning: Null handling, duplicates, data types

*Objective:* Clean missing values, cast types, deduplicate records.

*Theory:* na.fill/na.replace, dropna, coalesce, cast, distinct vs dropDuplicates.

In [0]:
raw = [
  ("1","Alice","34"),
  ("2","Bob",None),
  ("2","Bob",None),
  ("3","Cathy","29")
]
df_raw = spark.createDataFrame(raw, ["id","name","age_str"])

In [0]:
df_raw.show()

In [0]:
#cast
df_clean = (df_raw.withColumn("id",F.col("id").cast("int"))
             .withColumn("age",F.col("age_str").cast("int"))
             .drop("age_str"))
df_clean.show()

In [0]:
df_raw.printSchema()

In [0]:
df_clean.printSchema()

In [0]:
df_clean.show()

In [0]:
#null handling
df_clean = df_clean.na.fill({"age":-1,"name":"Unknown"})
df_clean.show()

In [0]:
#drop duplicates
df_unique = df_clean.dropDuplicates(["id","name"])
df_unique.show()

#### Find Duplicates


In [0]:
dup_df= (df_clean.groupBy("id") 
         .agg(F.count("id").alias("total_count"))
         .filter(F.col("total_count")>1))
dup_df.show()

In [0]:
df_clean.show()

In [0]:
from pyspark.sql.window import Window
windowSpec = Window.partitionBy('id').orderBy(F.desc('age'))
df_dup1 = df_clean.withColumn("row_number",F.row_number().over(windowSpec))
df_dup1.filter(F.col("row_number")>1).show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# sample transactions
tx = [
  (1, "2026-09-01", 50),
  (1, "2026-10-01", 100),
  (2, "2026-09-15", 200),
  (2, "2026-11-02", 25),
  (3, "2026-08-01", 10)
]
tx_df = spark.createDataFrame(tx, ["cust_id", "txn_date", "amount"])


In [0]:
display(tx_df)

In [0]:
# latest transaction per customer (row_number)
windowSpec = Window.partitionBy('cust_id').orderBy(F.desc('txn_date'))
latest_txn = tx_df.withColumn("row_number",F.row_number().over(windowSpec))
latest_txn.filter(F.col("row_number")==1).drop('row_number').show()

In [0]:
# top-3 transactions per customer by amount (rank vs dense_rank)
windowSpec = Window.partitionBy("cust_id").orderBy("amount")
top_cust = tx_df.withColumn("rank",F.rank().over(windowSpec)).withColumn("dense_rank",F.dense_rank().over(windowSpec))
top_cust.filter(F.col("rank")<=3).show()

In [0]:
%sql
-- create temp view first
CREATE OR REPLACE TEMP VIEW tx_v AS
SELECT * FROM VALUES
  (1, '2026-09-01', 50),
  (1, '2026-10-01', 100),
  (2, '2026-09-15', 200),
  (2, '2026-11-02', 25),
  (3, '2026-08-01', 10)
AS t(cust_id, txn_date, amount);

-- latest using row_number
SELECT cust_id,txn_date,amount 
FROM (
    SELECT *,row_number() OVER(PARTITION BY cust_id ORDER BY txn_date desc) as rn FROM tx_v
) as ranked
WHERE ranked.rn=1;


###UDF and Pandas UDF: Implement custom functions

In [0]:
from pyspark.sql.types import IntegerType
@F.udf(IntegerType())
def name_length(s):
    return len(s) if s else 0

names = [(1, "Alice"), (2, None), (3, "Bob")]
names_df = spark.createDataFrame(names, ["id", "name"])
names_df.withColumn("name_len", name_length(F.col("name"))).show()


### Create Temp View

In [0]:
sales = [(1,"2026-10-01",100.0),(1,"2026-10-15",250.0),(2,"2026-10-05",300.0)]
sales_df = spark.createDataFrame(sales, ["cust_id","date","amount"])
sales_df.createOrReplaceTempView("sales_v")

In [0]:
#Spark SQL
res = spark.sql("""
                SELECT cust_id,date_format(to_date(date,'yyyy-MM-dd'),'yyyy-MM') as month,sum(amount) as total_amount
                FROM sales_v
                GROUP BY cust_id,month
                ORDER BY cust_id,month
                """)
res.show()

In [0]:
# temp view: visible only in current SparkSession
df = spark.createDataFrame([(1,"a")], ["id","val"])
df.createOrReplaceTempView("tmp_v")
spark.sql("SELECT * FROM tmp_v").show()
